In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

# 1. 定义状态
class State(TypedDict):
    foo: str
    bar: list[str]

# 2. 定义节点
def node_a(state: State) -> State:
    print("▶ 执行 node_a")
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State) -> State:
    print("▶ 执行 node_b")
    return {"foo": "b", "bar": ["b"]}

# 3. 构建状态机
builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

# 流程（修复笔误：节点名用字符串）
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)  # 正确写法

# 4. 内存记忆存储
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

# ==============================================
# 执行 + 状态管理
# ==============================================
if __name__ == "__main__":
    # 对话ID（核心：区分用户）
    config = {"configurable": {"thread_id": "1"}}

    # 1. 首次执行
    print("=== 首次执行工作流 ===")
    result = graph.invoke({"foo": "", "bar": []}, config=config)
    print("执行结果：", result, "\n")

    # 2. 获取当前最新状态
    print("=== 获取最新状态 ===")
    current_state = graph.get_state(config)
    print("状态数据：", current_state.values, "\n")

    # 3. 获取所有历史检查点（顺序：最新 → 最早）
    print("=== 获取历史检查点 ===")
    state_history = list(graph.get_state_history(config))
    for idx, snapshot in enumerate(state_history):
        cp_id = snapshot.config["configurable"]["checkpoint_id"]
        print(f"检查点 {idx} | ID: {cp_id} | 状态: {snapshot.values}")

    # 4. 回溯历史状态（取 执行node_a之后 的状态，最直观）
    print("\n=== 回溯历史状态 ===")
    # 取索引1：执行完node_a的状态（非初始空状态）
    target_snapshot = state_history[1]
    historical_config = {
        "configurable": {
            "thread_id": "1",
            "checkpoint_id": target_snapshot.config["configurable"]["checkpoint_id"],
        }
    }
    # 获取历史状态
    historical_state = graph.get_state(historical_config)
    print("回溯到 node_a 执行后的状态：", historical_state.values)

=== 首次执行工作流 ===
▶ 执行 node_a
▶ 执行 node_b
执行结果： {'foo': 'b', 'bar': ['b']} 

=== 获取最新状态 ===
状态数据： {'foo': 'b', 'bar': ['b']} 

=== 获取历史检查点 ===
检查点 0 | ID: 1f1528eb-7847-66c8-8002-3344f15702df | 状态: {'foo': 'b', 'bar': ['b']}
检查点 1 | ID: 1f1528eb-783b-638b-8001-2199136d625b | 状态: {'foo': 'a', 'bar': ['a']}
检查点 2 | ID: 1f1528eb-7832-62dd-8000-3ce708f535e1 | 状态: {'foo': '', 'bar': []}
检查点 3 | ID: 1f1528eb-782d-64e8-bfff-86d5a302f0b5 | 状态: {}

=== 回溯历史状态 ===
回溯到 node_a 执行后的状态： {'foo': 'a', 'bar': ['a']}


In [7]:
config = {"configurable": {"thread_id": "1"}}
state_history = list(graph.get_state_history(config))

# 🔥 直接换行打印列表（核心：join + 格式化）
print("=== 历史检查点 ===")
print('\n' + '-'*60 + '\n'.join(
    [f"检查点 {i}:\nID: {s.config['configurable']['checkpoint_id']}\n状态: {s.values}" 
     for i, s in enumerate(state_history)]
))

=== 历史检查点 ===

------------------------------------------------------------检查点 0:
ID: 1f1528eb-7847-66c8-8002-3344f15702df
状态: {'foo': 'b', 'bar': ['b']}
检查点 1:
ID: 1f1528eb-783b-638b-8001-2199136d625b
状态: {'foo': 'a', 'bar': ['a']}
检查点 2:
ID: 1f1528eb-7832-62dd-8000-3ce708f535e1
状态: {'foo': '', 'bar': []}
检查点 3:
ID: 1f1528eb-782d-64e8-bfff-86d5a302f0b5
状态: {}


In [8]:
%pip install deepagents tavily-python

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ---------------------------------------- 0.0/763.1 kB ? eta -:--:--
     ---------------------------------------- 763.1/763.1 kB 16.2 MB/s  0:00:00
     ---------------------------------------- 0.0/793.7 kB ? eta -:--:--
     ---------------------------------------- 793.7/793.7 kB 16.6 MB/s  0:00:00

   ----------------------------------------  0/14 [filetype]
   ----------------------------------------  0/14 [filetype]
   ----------------------------------------  0/14 [filetype]
   ----------------------------------------  0/14 [filetype]
   ----------------------------------------  0/14 [filetype]
   ----------------------------------------  0/14 [filetype]
   -- -------------------------------------  1/14 [websockets]
   -- -------------------------------------  1/14 [websockets]
   -- -------------------------------------  1/14 [websockets]
   -- -------------------------------------  1/14 [websockets]
   -- ---------